# Anomaly Detection — Colab Demo

Unsupervised anomaly detection on **SMD** with **TimesNet, TimeMixer, PatchTST and ModernTCN**.

| Runtime | SMD · win=100 · 10 epochs · step=10 |
|---------|--------------------------------------|
| T4 GPU  | ~10–30 min                           |
| CPU     | ~2–6 h                               |

> **Tip:** Runtime → Change runtime type → T4 GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
!git clone https://github.com/caltinuzengi/Time-Series-Library.git
%cd Time-Series-Library/

In [ ]:
!git pull
!pip install uv -q
!uv sync

In [ ]:
!uv run python scripts/download_anomaly_data.py

In [ ]:
!bash scripts/TimesNet/debug_anomaly_SMD.sh

In [ ]:
!bash scripts/TimeMixer/debug_anomaly_SMD.sh

In [ ]:
!bash scripts/PatchTST/debug_anomaly_SMD.sh

In [ ]:
!bash scripts/ModernTCN/debug_anomaly_SMD.sh

In [ ]:
!bash scripts/TimesNet/anomaly_SMD.sh

In [ ]:
!bash scripts/TimeMixer/anomaly_SMD.sh

In [ ]:
!bash scripts/PatchTST/anomaly_SMD.sh

In [ ]:
!bash scripts/ModernTCN/anomaly_SMD.sh

## 📊 Sonuçlar

Aşağıdaki hücreler `logs/` ve `results/` klasörlerindeki JSON dosyalarını okur — önce yukarıdaki eğitim hücrelerini çalıştırın.

In [ ]:
import json, glob
import matplotlib.pyplot as plt

models = ["TimesNet", "TimeMixer", "PatchTST", "ModernTCN"]
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)

for ax, model in zip(axes, models):
    files = sorted(glob.glob(f"logs/{model}_SMD_anomaly_detection_100_*.json"))
    if not files:
        ax.set_title(f"{model}\n(log yok)")
        continue
    with open(files[-1]) as f:
        d = json.load(f)
    logs = d["epoch_logs"]
    epochs = [e["epoch"] for e in logs]
    ax.plot(epochs, [e["train_loss"] for e in logs], "o-", color="#1f77b4")
    ax.set_title(model)
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Recon MSE")
plt.suptitle("SMD — Anomaly Detection Training Loss")
plt.tight_layout()
plt.show()

In [ ]:
import json, glob, pandas as pd

rows = []
for path in sorted(glob.glob("results/*_SMD_anomaly_detection_*.json")):
    with open(path) as f:
        d = json.load(f)
    cfg, met = d["config"], d["metrics"]
    rows.append({
        "Model":      cfg.get("model", "-"),
        "F1":         round(met.get("f1",          float("nan")), 4),
        "F1-PA":      round(met.get("f1_pa",        float("nan")), 4),
        "Precision":  round(met.get("precision",    float("nan")), 4),
        "Recall":     round(met.get("recall",       float("nan")), 4),
        "AUROC":      round(met.get("auroc",        float("nan")), 4),
    })

if rows:
    df = pd.DataFrame(rows).sort_values("F1-PA", ascending=False)
    display(df)
else:
    print("Henüz sonuç yok — önce eğitim hücrelerini çalıştırın.")